# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [4]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [5]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [6]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [7]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [1]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'capabilities page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'capabilities page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog / insights', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


{'links': [{'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://huggingface.co/join'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
about 5 hours ago
•
138k
•
2.97k
moonshotai/Kimi-K2.6
Updated
4 days ago
•
443k
•
1.08k
Qwen/Qwen3.6-27B
Updated
3 days ago
•
399k
•
884
openai/privacy-filter
Updated
5 days ago
•
47.5k
•
884
deepseek-ai/DeepSeek-V4-Flash
Updated
about 5 hours ago
•
65.7k
•
761
Browse 2M+ models
Spaces
Running
on
CPU Upgrade
200
ML Intern
🤖
200
Chat with an AI “ML Intern” to get quick ML help
Running
on
Zero
MCP
2.4k
Wan2.2 14B Preview
🐌
2.4k
generate a video from an image with a text prompt
Running
on
Zero
MCP
Featured
1.03k
FireRed Image Edit 1.0 Fast
🌖
1.03k
FireRed-Image-Edit × Qwen-Image-Edit-Rapid (Tran

In [20]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [21]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\nabout 5 hours ago\n•\n138k\n•\n2.97k\nmoonshotai/Kimi-K2.6\nUpdated\n4 days ago\n•\n443k\n•\n1.08k\nopenai/privacy-filter\nUpdated\n5 days ago\n•\n47.5k\n•\n885\nQwen/Qwen3.6-27B\nUpdated\n3 days ago\n•\n399k\n•\n884\ndeepseek-ai/DeepSeek-V4-Flash\nUpdated\nabout 5 hours ago\n•\n65.7k\n•\n761\nBrowse 2M+ models\nSpaces\nRunning\non\nCPU Upgrade\n200\

In [16]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a vibrant AI community and collaboration platform dedicated to building the future of machine learning. It serves as the central hub where machine learning engineers, scientists, and enthusiasts from around the world come together to create, discover, and share models, datasets, and AI applications.

The platform supports a broad range of AI modalities including text, image, video, audio, and even 3D data. With over 2 million models and more than 500,000 datasets available, Hugging Face enables rapid development and deployment of cutting-edge machine learning technologies.

---

## What We Offer

- **Open Collaboration:** Host and collaborate on unlimited public models, datasets, and applications.
- **Diverse Modalities:** Build and explore AI models for text, images, videos, audio, and 3D applications.
- **AI Applications:** Access over 1 million AI-powered applications and trending projects like voice cloning in 600+ languages, video generation from images, and local web-based LLMs.
- **Community & Support:** Engage with an active, fast-growing global community to learn, share, and innovate.
- **Open Source Stack:** Accelerate your machine learning projects with our open-source tools and resources.
- **Enterprise Solutions:** Scalable solutions and paid compute plans tailored for business needs.

---

## Our Community and Customers

Hugging Face empowers developers, researchers, startups, and enterprises to build open and ethical AI solutions collaboratively. The platform is widely used by:

- **Machine Learning Engineers & Researchers:** To host, discover, and collaborate on the latest models and datasets.
- **Startups & Enterprises:** To accelerate product development and incorporate AI-powered functionalities efficiently.
- **AI Enthusiasts & Educators:** To learn, experiment, and build portfolios with state-of-the-art models.
- **Global Developers:** Enabling access to user-friendly tools such as chatbots and AI assistants for quick ML help.

---

## Company Culture

At Hugging Face, we believe in openness, collaboration, and ethical AI. Our culture fosters:

- **Community-Driven Innovation:** We put the community first, encouraging sharing and collaborative problem solving.
- **Transparency:** Commitment to open-source principles and accessible AI development.
- **Diversity & Inclusion:** Welcoming contributors of all backgrounds and skill levels.
- **Continuous Learning:** Supporting growth through resources, shared knowledge, and hands-on experimentation.

---

## Careers at Hugging Face

Join us in shaping the future of AI!

We are constantly looking for passionate individuals who want to:

- Develop next-generation AI tools and infrastructure.
- Contribute to open-source machine learning projects.
- Engage with a global community of AI professionals.
- Bring ethical and innovative AI products to life.

If you are excited by cutting-edge AI technology and community collaboration, explore opportunities with Hugging Face to grow your career while making an impact.

---

## Get Started with Hugging Face

- **Browse 2M+ Models:** Find models for NLP, computer vision, audio processing, and more.
- **Explore 1M+ AI Applications:** Discover tools and demos built by the community.
- **Contribute:** Share your own models, datasets, and applications.
- **Accelerate Your ML Projects:** Access paid compute and enterprise features as needed.
- **Join the Community:** Sign up to build your profile, showcase your work, and collaborate.

Visit [huggingface.co](https://huggingface.co) to join the home of machine learning innovation.

---

## Brand Highlights

- Signature colors: Bright yellow (#FFD21E), vibrant orange (#FF9D00), and neutral gray (#6B7280).
- Recognized logos available in SVG, PNG, and AI formats symbolizing openness and friendliness.

---

**Hugging Face** — The AI community building the future, together.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future — a leading collaboration platform where the machine learning (ML) community worldwide comes together to create, discover, and collaborate on ML models, datasets, and applications. It serves as a central hub, empowering engineers, scientists, and users to share and experiment with open-source AI technology, fostering an open and ethical AI ecosystem.

---

## Platform Highlights

- **Extensive Model Library:** Access and browse over **2 million models**, spanning multiple modalities including text, image, video, audio, and even 3D.
- **Vibrant Dataset Repository:** Explore from over **500,000 datasets**, continuously updated by the community and major organizations such as NVIDIA.
- **Spaces & Apps:** Run and share AI apps and interactive demos powered by state-of-the-art ML models, running smoothly on various computational backends.
- **Open Source Stack:** Leverage Hugging Face’s open source tools to accelerate ML development, including transformers, tokenizers, datasets, and more.
- **Community Powered:** Engage with a fast-growing global community contributing diverse models, datasets, and applications.

---

## Why Choose Hugging Face?

- **Collaboration at Scale:** Host, share, and collaborate on unlimited public models, datasets, and applications with ease.
- **Multi-Modal AI:** Build and experiment across various media types—text, image, video, audio, and 3D applications.
- **Build Your Portfolio:** Showcase your projects and skills to the global ML community, enhancing your professional profile.
- **Accelerate Innovation:** Use optimized infrastructure and paid compute options tailored to your project needs, speeding up your development lifecycle.
- **Ethical AI Commitment:** Join a platform dedicated to open, transparent, and ethical AI development aligned with community values.

---

## Our Customers & Users

Hugging Face serves a broad spectrum of users including:

- Machine Learning Engineers and Researchers looking for accessible shared resources.
- Data Scientists and Developers building production-ready AI applications.
- Organizations and Enterprises integrating AI safely and effectively via Hugging Face Enterprise solutions.
- Academic and Industry Communities using open-source models to push the boundaries of AI research.
- Newcomers and enthusiasts who want to learn and contribute to growing AI projects.

---

## Careers at Hugging Face

Hugging Face fosters a culture of openness, collaboration, and innovation. Joining the team means:

- Working alongside passionate AI experts and contributors globally.
- Contributing to a mission-driven company focused on creating an open, ethical AI future.
- Access to cutting-edge projects in AI and ML, with opportunities to influence industry standards.
- Flexible, inclusive environment promoting continuous learning and growth.

**Current Opportunities:** Hugging Face hires across technical, research, and operational roles. Prospective candidates interested in AI, open source software, and building community-driven solutions are encouraged to apply via their website.

---

## Get Involved

- **Explore:** Browse their massive library of models and datasets.
- **Contribute:** Share your own models, datasets, or AI applications.
- **Collaborate:** Join discussions, contribute to open-source projects, or participate in community events.
- **Build:** Utilize Hugging Face’s tools and infrastructure to accelerate your AI projects.

---

## Connect with Hugging Face

- Visit: [huggingface.co](https://huggingface.co)
- Join the Community: Collaborate and connect with millions of AI enthusiasts.
- Explore Resources: Docs, tutorials, and enterprise offerings available on their site.

---

Hugging Face — powering the future of AI through community, collaboration, and innovation. Join us and shape the next era of machine learning!

In [22]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Welcome to Hugging Face: Where AI Meets Community (and a Hug or Two)

---

## Who We Are

Hugging Face isn’t just a company; we’re the ultimate AI hangout spot — a vibrant community where machine learning maestros, curious newcomers, and everything in-between come together to *build the future.* Think of us as the "Coachella" of AI models, datasets, and applications… but with fewer sunburns and way more algorithms. 🌞🤖

---

## What We Offer

- **2 Million+ Models**  
  From deepseek-ai’s latest moonshot to openai’s privacy filters, our model hub is the AI equivalent of a Netflix binge—only smarter and less addictive (we hope).

- **500k+ Datasets**  
  For every problem, an open dataset ready to fuel your next breakthrough—or at least impress your boss.

- **Spaces — AI Playground for Pros & Noobs Alike**  
  Build, share, and run ML apps in minutes. Want to chat with an AI “ML Intern” for quick help? We got it. Generate videos from images with text prompts? Yup. Tinker with local browser-based LLMs? Absolutely. Basically, your AI sandbox with no sand in your shoes.

- **Buckets**  
  Because what’s a cool AI hub without some serious cloud storage, right?

---

## For Enterprise & Teams

Think of us as your AI fairy godmother, waving the wand to scale your organization’s AI prowess with:  

- Enterprise-grade security (because your AI secrets are sacred).
- Single Sign-On & granular access controls to keep the right folks in the loop.
- Audit logs, billing controls, and priority support—because adulting in AI needs structure.
- 5x boosts in ZeroGPU quota to power up your magic.
- Private datasets & storage to keep your crown jewels safe.

All starting at just $20 per user/month — cheaper than your daily coffee, way more caffeinating.

---

## Company Culture: More Hugs, Less Debugs

At Hugging Face, we’re passionate about collaboration, open source, and having fun while we’re at it. Whether you’re a machine learning whiz, a data enthusiast, or someone just curious about the AI world, you’re invited to build, share, and learn with us.

We believe in:  
- Open and ethical AI  
- Inclusivity and community spirit  
- Encouraging creativity through sharing (yes, even weird experiments!)  
- Building a portfolio that makes your future self proud  

---

## Careers: Join the AI Revolution

Want to be part of the team that’s *literally* hugging the future of AI? We’re looking for:

- Developers who dream in code and occasionally in emojis  
- Researchers obsessed with pushing AI limits  
- Community wranglers who keep the conversations flowing  
- Enterprise heroes who help clients scale with smiles  

Perks include working alongside the brightest minds in ML, contributing to open source projects with global impact, and yes — a culture that values your quirks as much as your skills.

---

## Why Hugging Face?

Because here, AI isn’t just tech — it’s a global community. We empower the next generation of machine learning engineers, scientists, and end-users to **share, explore, and experiment** with open source like never before.

Oh, and did we mention the color palette? #FFD21E yellow vibes for that sunny optimism, #FF9D00 for fiery innovation, and #6B7280 because even AI likes to dress in sleek gray.

---

## Ready to Hug the Future?

- **Explore 2M+ Models**  
- **Browse 500k+ Datasets**  
- **Make AI Apps in Minutes**  
- **Join Teams & Enterprises Scaling AI Worldwide**  

Sign up today and start building the future — because the AI world needs more hugs, less bugs.

---

*Hugging Face: The AI community where brains, bytes, and hugs collide.* 🤗✨

---

*Psst… Want the logos, colors, and branding magic? Download them from our Brand Assets page and rock that Hugging Face swag!*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>